# P8 — Notebook 1: Compute ρ_k (Band-wise Spectral Preservation)

**Project**: P8 — Fourier-Theoretic Account of SPH → IEEE TIP submission M5 (Oct 2026)
**Author**: Vo Thanh Kiet (VSB-TUO) | **Date**: 2026-05-14 | **Version**: v1

## Goal
Compute the multi-band spectral preservation ratio
$$\rho_k^{(m, \sigma, d)} = 1 - \min\left(1, \frac{|E_k(D_m(I_\sigma)) - E_k(I_{\text{clean}})|}{E_k(I_{\text{clean}})}\right)$$
for K=4 radial bands, 5 denoising methods, 5 noise levels, 3 datasets.

## Plan (12 cells, each with verify check)
1. Setup + GPU check
2. BFPI library (inline)
3. Config (paths, K=4 bands, Phase A=50 imgs/dataset)
4. Load denoise models (.pt files from PnPLO Drive)
5. Helper: load test images per dataset
6. Helper: apply denoising (BM3D library + deep models)
7. **Idempotent guard** — skip if output exists
8. Compute ρ_k batch loop (Phase A subsample)
9. Save to Drive
10. Sanity checks against P6 paper
11. Phase A summary + Phase B decision point
12. (Conditional) Phase B full set

## Output deliverable
`/content/drive/MyDrive/.../P8_outputs/rho_per_cell__phaseA__v1.csv`

---
## Cell 1 — Setup + GPU check

In [ ]:
# Cell 1: Setup
import os
import sys
import time

# Mount Drive
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
print('✓ Drive mounted')

# GPU check
gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f'GPU: {gpu_info[0] if gpu_info else "NONE"}')

# Install deps (BM3D)
try:
    import bm3d
    print('✓ bm3d already installed')
except ImportError:
    !pip install bm3d --quiet
    import bm3d
    print('✓ bm3d installed')

# Verify
import numpy as np
import pandas as pd
import torch
print(f'  Python {sys.version_info.major}.{sys.version_info.minor}, '
      f'numpy {np.__version__}, torch {torch.__version__}, '
      f'cuda available: {torch.cuda.is_available()}')

---
## Cell 2 — BFPI library (inline for portability)

Verify against `bfpi_lib_v02.py` from local Barry session. Same code, same semantics.

In [ ]:
# Cell 2: BFPI library (inline) — P8 v0.3 (phase-aware)
"""
Multi-band spectral preservation library — P8 v0.3.
Base: bfpi_lib_v02.py (Barry 2026-05-14); v0.3 extension per P8_math_v03.md.

v0.3 changes:
  - get_band_boundaries: LAST band unbounded -> covers DFT corners R in (pi, sqrt2*pi].
  - NEW band_complex_distance / compute_rho_phase: PHASE-AWARE preservation
        rho_phase_k = 1 - ||F_k(clean) - F_k(denoised)|| / ||F_k(clean)||
    This is the quantity used by the Theorem 1 bound. The old energy 'sim'
    mode is kept only for backward-compatible descriptive columns.
"""
import numpy as np
from scipy import fft as sp_fft
from typing import Optional


def get_band_boundaries(K: int = 4, scheme: str = 'linear') -> np.ndarray:
    """K+1 radial boundaries. v0.3: last boundary = inf (unbounded last band)."""
    if scheme == 'linear':
        b = np.linspace(0, np.pi, K + 1)
        b[-1] = np.inf
        return b
    raise ValueError(f'Unknown scheme: {scheme}')


def radial_frequency_map(H: int, W: int) -> np.ndarray:
    """Build 2D radial frequency map for HxW image."""
    fy = np.fft.fftshift(np.fft.fftfreq(H)) * 2 * np.pi
    fx = np.fft.fftshift(np.fft.fftfreq(W)) * 2 * np.pi
    fxx, fyy = np.meshgrid(fx, fy, indexing='xy')
    return np.sqrt(fxx**2 + fyy**2)


def band_energy(image: np.ndarray, boundaries: np.ndarray,
                radial_map: Optional[np.ndarray] = None) -> np.ndarray:
    """Spectral energy per radial band ||F_k||^2."""
    if image.ndim == 3:
        return np.stack([band_energy(image[..., c], boundaries, radial_map)
                         for c in range(image.shape[-1])]).mean(axis=0)
    H, W = image.shape
    if radial_map is None:
        radial_map = radial_frequency_map(H, W)
    F = np.fft.fftshift(sp_fft.fft2(image.astype(np.float64)))
    power = np.abs(F) ** 2
    K = len(boundaries) - 1
    E = np.zeros(K)
    for k in range(K):
        mask = (radial_map >= boundaries[k]) & (radial_map < boundaries[k + 1])
        E[k] = power[mask].sum()
    return E


def band_complex_distance(clean: np.ndarray, denoised: np.ndarray,
                          boundaries: np.ndarray,
                          radial_map: Optional[np.ndarray] = None) -> np.ndarray:
    """P8 v0.3: per-band squared COMPLEX-spectral distance ||F_k(clean)-F_k(den)||^2.
    Phase-aware: uses the full complex DFT, not magnitude only."""
    if clean.ndim == 3:
        return np.stack([band_complex_distance(clean[..., c], denoised[..., c],
                         boundaries, radial_map)
                         for c in range(clean.shape[-1])]).mean(axis=0)
    H, W = clean.shape
    if radial_map is None:
        radial_map = radial_frequency_map(H, W)
    Fc = np.fft.fftshift(sp_fft.fft2(clean.astype(np.float64)))
    Fd = np.fft.fftshift(sp_fft.fft2(denoised.astype(np.float64)))
    dpow = np.abs(Fc - Fd) ** 2
    K = len(boundaries) - 1
    D = np.zeros(K)
    for k in range(K):
        mask = (radial_map >= boundaries[k]) & (radial_map < boundaries[k + 1])
        D[k] = dpow[mask].sum()
    return D


def compute_rho_phase(clean: np.ndarray, denoised: np.ndarray,
                      boundaries: np.ndarray,
                      radial_map: Optional[np.ndarray] = None,
                      E_clean: Optional[np.ndarray] = None,
                      eps: float = 1e-10) -> np.ndarray:
    """P8 v0.3 phase-aware band-wise spectral preservation (Theorem 1 quantity):
        delta_k = ||F_k(clean) - F_k(den)|| / ||F_k(clean)||
        rho_phase_k = 1 - delta_k    (<= 1; may be < 0 under heavy distortion).
    """
    if radial_map is None:
        radial_map = radial_frequency_map(*clean.shape[:2])
    if E_clean is None:
        E_clean = band_energy(clean, boundaries, radial_map)
    D = band_complex_distance(clean, denoised, boundaries, radial_map)
    return 1.0 - np.sqrt(D / (E_clean + eps))


def compute_rho_k(clean: np.ndarray, denoised: np.ndarray,
                  boundaries: np.ndarray,
                  radial_map: Optional[np.ndarray] = None,
                  eps: float = 1e-10, mode: str = 'sim') -> np.ndarray:
    """Legacy energy-based quantity (kept for descriptive columns only).
    mode='sim' : 1 - min(1, |E_den - E_clean| / E_clean)  in [0,1]
    mode='ratio': E_den / E_clean  in [0, inf)  -- two-mode diagnostic.
    """
    if radial_map is None:
        radial_map = radial_frequency_map(*clean.shape[:2])
    E_clean = band_energy(clean, boundaries, radial_map)
    E_denoised = band_energy(denoised, boundaries, radial_map)
    if mode == 'sim':
        gap = np.abs(E_denoised - E_clean) / (E_clean + eps)
        return 1.0 - np.minimum(1.0, gap)
    elif mode == 'ratio':
        return E_denoised / (E_clean + eps)
    raise ValueError(f'Unknown mode: {mode}')


# --- Sanity self-test (v0.3) — HARD ASSERT, fail fast ---
_b = get_band_boundaries(K=4)
_rng = np.random.RandomState(42)
_img = _rng.rand(64, 64) * 255
assert np.allclose(compute_rho_k(_img, _img, _b, mode='sim'), 1.0, atol=1e-6)
assert np.allclose(compute_rho_phase(_img, _img, _b), 1.0, atol=1e-9)
assert np.isinf(_b[-1])
print('\u2713 BFPI v0.3 loaded & self-tested: rho_sim & rho_phase identity OK; '
      'last band unbounded.')


---
## Cell 3 — Config

**Phase A**: subsample 50 imgs/dataset (quick validation, ~30min compute)
**Phase B**: full test sets (paper-grade, ~5h compute, only if Phase A passes)

All paths verified against output.txt of inspection script.

In [ ]:
# Cell 3: Config
from pathlib import Path

# === Project root on Drive ===
DRIVE_ROOT = Path('/content/drive/MyDrive/DOCTOR_PHD/FINAL PROJECT')

# === 3 Datasets ===
DATASETS = {
    'PnPLO': {
        # P8 v1.1 FIX: use TRAIN split (944 imgs, per Kiet's clarification — val/test are for detector eval, not image metrics)
        'clean_dir': DRIVE_ROOT / '04_RESULT_TRAIN_KARTHY/YOLO_Denoise_Experiment_Karthy/dataset_yolo/images/train',
        'denoise_models_dir': DRIVE_ROOT / '04_RESULT_TRAIN_KARTHY/YOLO_Denoise_Experiment_Karthy/yolo_denoise_experiment/denoise_models',
        'output_dir': DRIVE_ROOT / '04_RESULT_TRAIN_KARTHY/P8_outputs',
    },
    'VOC': {
        # P8 v1.1 FIX: use TRAIN split (per Kiet, val/test are for detector evaluation, not image-level metrics)
        'clean_dir': DRIVE_ROOT / '04_1_VOC_experiment/VOC_person/dataset_yolo/train/images',
        'denoise_models_dir': None,  # Use PnPLO models (Option A locked)
        'output_dir': DRIVE_ROOT / '04_1_VOC_experiment/P8_outputs',
    },
    'INRIA': {
        # P8 v1.1 FIX: use TRAIN split (632 imgs)
        'clean_dir': DRIVE_ROOT / '04_2_INRIA_experiment/INRIA Person/train/images',
        'denoise_models_dir': None,
        'output_dir': DRIVE_ROOT / '04_2_INRIA_experiment/P8_outputs',
    },
}

# === Experiment factors ===
DENOISE_METHODS = ['bm3d', 'gaussian_filter', 'autoencoder', 'dncnn', 'cae_pso']
SIGMAS = [1, 5, 10, 20, 30]  # exclude sigma=0 (clean=denoised trivially)
K_BANDS = 4

# === Phase A: subsample ===
N_IMAGES_PER_DATASET = 50  # Phase A
RANDOM_SEED = 42

# === Output paths (per-dataset, P8 v1.1) ===
# Each dataset has its own P8_outputs/ folder under its experiment root.
# This keeps each dataset folder self-contained for archiving/sharing.
OUTPUT_CSVS = {}
for ds_name, ds_conf in DATASETS.items():
    out_dir = ds_conf['output_dir']
    out_dir.mkdir(parents=True, exist_ok=True)
    OUTPUT_CSVS[ds_name] = out_dir / f'rho_per_cell__{ds_name}__phaseA__v1.csv'
    print(f'  {ds_name} output: {OUTPUT_CSVS[ds_name]}')

# === Verify all paths exist ===
print('\nVerifying paths:')
all_ok = True
for ds_name, ds_conf in DATASETS.items():
    clean = ds_conf['clean_dir']
    status = '✓' if clean.exists() else '❌'
    print(f'  {status} {ds_name} clean: {clean}')
    if not clean.exists():
        all_ok = False

pnplo_models = DATASETS['PnPLO']['denoise_models_dir']
status = '✓' if pnplo_models.exists() else '❌'
print(f'  {status} PnPLO denoise models: {pnplo_models}')
if not pnplo_models.exists():
    all_ok = False

assert all_ok, 'FAIL: not all paths exist — fix Config before proceeding'
print('\n✓ All paths verified')

---
## Cell 4 — Load PnPLO denoise models

Per inspection output, PnPLO has 15 model files: `{autoencoder, cae_pso, dncnn}_sigma{1, 5, 10, 20, 30}.pt`. Per Option A, we use PnPLO models for all 3 datasets (cross-domain denoising assumption).

For BM3D and gaussian_filter: no .pt needed (parameter-free).

In [ ]:
# Cell 4: Load denoise models (P8 v3 — instantiate architectures + load_state_dict)
import torch
import torch.nn as nn
import yaml

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

denoise_models_dir = DATASETS['PnPLO']['denoise_models_dir']
DEEP_METHODS = ['autoencoder', 'dncnn', 'cae_pso']


# === Architecture definitions (verbatim from training notebook Cells 10-12) ===

class DenoisingAutoencoder(nn.Module):
    """Convolutional Denoising Autoencoder (training notebook Cell 10)."""
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 128, 2, stride=2), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 64, 2, stride=2), nn.ReLU(inplace=True),
            nn.Conv2d(64, 3, 3, padding=1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))


class DnCNN(nn.Module):
    """DnCNN with residual learning (training notebook Cell 11)."""
    def __init__(self, channels=3, num_layers=17, features=64):
        super().__init__()
        layers = [nn.Conv2d(channels, features, 3, padding=1, bias=False),
                  nn.ReLU(inplace=True)]
        for _ in range(num_layers - 2):
            layers += [nn.Conv2d(features, features, 3, padding=1, bias=False),
                       nn.BatchNorm2d(features),
                       nn.ReLU(inplace=True)]
        layers.append(nn.Conv2d(features, channels, 3, padding=1, bias=False))
        self.layers = nn.Sequential(*layers)
    def forward(self, x):
        return x - self.layers(x)


class CAE(nn.Module):
    """Configurable CAE optimized by PSO (training notebook Cell 12)."""
    def __init__(self, base_filters=64, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        f1, f2 = base_filters, base_filters * 2
        self.encoder = nn.Sequential(
            nn.Conv2d(3, f1, kernel_size, padding=pad), nn.BatchNorm2d(f1), nn.ReLU(inplace=True),
            nn.Conv2d(f1, f1, kernel_size, padding=pad), nn.BatchNorm2d(f1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(f1, f2, kernel_size, padding=pad), nn.BatchNorm2d(f2), nn.ReLU(inplace=True),
            nn.Conv2d(f2, f2, kernel_size, padding=pad), nn.BatchNorm2d(f2), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(f2, f2, 2, stride=2), nn.BatchNorm2d(f2), nn.ReLU(inplace=True),
            nn.Conv2d(f2, f1, kernel_size, padding=pad), nn.BatchNorm2d(f1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(f1, f1, 2, stride=2), nn.BatchNorm2d(f1), nn.ReLU(inplace=True),
            nn.Conv2d(f1, 3, kernel_size, padding=pad), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))


def _infer_cae_params(state_dict):
    """Suy ra base_filters và kernel_size từ encoder.0.weight shape."""
    w = state_dict.get('encoder.0.weight')
    if w is not None:
        return {'base_filters': w.shape[0], 'kernel_size': w.shape[2]}
    return {'base_filters': 64, 'kernel_size': 3}


def _build_model(method: str, state_dict):
    """Instantiate matching architecture and load weights."""
    if method == 'autoencoder':
        m = DenoisingAutoencoder()
    elif method == 'dncnn':
        m = DnCNN(channels=3, num_layers=17, features=64)
    elif method == 'cae_pso':
        p = _infer_cae_params(state_dict)
        m = CAE(base_filters=p['base_filters'], kernel_size=p['kernel_size'])
    else:
        raise ValueError(f'Unknown method: {method}')
    m.load_state_dict(state_dict)
    return m.eval().to(device)


# Eager-load all 15 models (5 sigmas × 3 methods). Total memory < 200MB, fine on A100.
denoise_models = {m: {} for m in DEEP_METHODS}
for method in DEEP_METHODS:
    for sigma in SIGMAS:
        pt_path = denoise_models_dir / f'{method}_sigma{sigma}.pt'
        if not pt_path.exists():
            print(f'  ❌ MISSING: {method}_sigma{sigma}.pt')
            continue
        size_mb = pt_path.stat().st_size / 1e6
        # weights_only=True is safe (state_dicts are pure tensors). Falls back if needed.
        try:
            state_dict = torch.load(pt_path, map_location=device, weights_only=True)
        except Exception:
            state_dict = torch.load(pt_path, map_location=device, weights_only=False)
        model = _build_model(method, state_dict)
        denoise_models[method][sigma] = model
        n_params = sum(p.numel() for p in model.parameters())
        extra = ''
        if method == 'cae_pso':
            p = _infer_cae_params(state_dict)
            extra = f"  [base_filters={p['base_filters']}, kernel_size={p['kernel_size']}]"
        print(f'  ✓ {method}_sigma{sigma}.pt  ({size_mb:.1f} MB, {n_params/1e6:.2f}M params){extra}')

# Verify counts
total_expected = len(DEEP_METHODS) * len(SIGMAS)
total_found = sum(len(v) for v in denoise_models.values())
assert total_found == total_expected, f'FAIL: loaded {total_found}/{total_expected} models'
print(f'\n✓ {total_found}/{total_expected} deep denoise models instantiated + loaded + on {device}')


---
## Cell 5 — Helper: load test images per dataset

Subsample N=50 images per dataset (Phase A). Use fixed seed for reproducibility.

In [ ]:
# Cell 5: Load test images (P8 v3 — dual: grayscale for BFPI + RGB for deep methods)
from PIL import Image
import random


def _rgb_to_luminance(rgb: np.ndarray) -> np.ndarray:
    """ITU-R BT.601 luminance."""
    return 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]


def load_images_dual(folder: Path, n_subsample: int, seed: int = 42):
    """
    Load up to n_subsample images, return TWO parallel dicts:
      - images_gray: {filename: gray_2d}  — for BFPI computation (P8 convention)
      - images_rgb:  {filename: rgb_3d}   — for deep denoisers (native training domain)
    Both float64, range [0, 255]. Same filenames in both.
    """
    files = sorted([f for f in folder.iterdir()
                     if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])
    print(f'  Found {len(files)} images in {folder.name}/')

    if len(files) > n_subsample:
        rng = random.Random(seed)
        files = rng.sample(files, n_subsample)
        files.sort()  # stable order after sampling

    images_gray, images_rgb = {}, {}
    for f in files:
        img_rgb = np.array(Image.open(f).convert('RGB'), dtype=np.float64)
        images_rgb[f.name]  = img_rgb
        images_gray[f.name] = _rgb_to_luminance(img_rgb)
    return images_gray, images_rgb


print('Loading clean images (grayscale + RGB pairs):')
clean_images = {}      # gray dict, preserved name for backward-compat with Cell 8 loop
clean_images_rgb = {}  # NEW: RGB companion
for ds_name, ds_conf in DATASETS.items():
    print(f'\n--- {ds_name} ---')
    g, r = load_images_dual(ds_conf['clean_dir'], N_IMAGES_PER_DATASET, RANDOM_SEED)
    clean_images[ds_name] = g
    clean_images_rgb[ds_name] = r
    assert len(g) == len(r) and len(g) > 0, f'FAIL: gray/RGB mismatch for {ds_name}'
    sg = next(iter(g.values()))
    sr = next(iter(r.values()))
    print(f'  → loaded {len(g)} imgs, gray shape={sg.shape}, rgb shape={sr.shape}')

# Summary
total_imgs = sum(len(v) for v in clean_images.values())
print(f'\n✓ Total clean images loaded: {total_imgs} (target: {N_IMAGES_PER_DATASET * 3})')
print(f'  Memory note: Phase A 50 imgs × 3 datasets ≈ 2 GB (gray+rgb). Fine on Colab.')
print(f'  Phase B full datasets (944+6340+632) may need chunked loading — revisit in Notebook 1 v4.')


---
## Cell 6 — Helpers: add noise + apply denoising

Two-stage pipeline:
1. **Add Gaussian noise** to clean image at level σ (deterministic per filename)
2. **Apply denoiser**:
   - `bm3d`: library call with oracle σ
   - `gaussian_filter`: 5×5 kernel, σ_g=1.0 (matches P6 setup)
   - `autoencoder` / `dncnn` / `cae_pso`: load .pt, forward pass on GPU

In [ ]:
# Cell 6: Denoising helpers (P8 v3 — Option C hybrid: methods native, BFPI on luminance)
import bm3d as bm3d_lib
from scipy import ndimage as ndi
import hashlib


def deterministic_seed(filename: str, sigma: int) -> int:
    """Per-image deterministic seed for noise reproducibility."""
    h = hashlib.md5(f'{filename}_{sigma}'.encode()).hexdigest()
    return int(h[:8], 16)


def add_gaussian_noise(clean: np.ndarray, sigma: int, filename: str) -> np.ndarray:
    """Add AWGN seeded per (filename, sigma). Works on gray 2D or RGB 3D."""
    rng = np.random.RandomState(deterministic_seed(filename, sigma))
    noise = rng.randn(*clean.shape) * sigma
    return np.clip(clean + noise, 0, 255)


def rgb_to_luminance(rgb: np.ndarray) -> np.ndarray:
    """ITU-R BT.601 luminance (matches Cell 5 convention)."""
    return 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]


# === Classical denoisers (operate on grayscale 2D) ===

def denoise_bm3d(noisy_gray: np.ndarray, sigma: int) -> np.ndarray:
    """BM3D on grayscale with oracle sigma."""
    out = bm3d_lib.bm3d(noisy_gray, sigma_psd=sigma)
    return np.clip(out, 0, 255)


def denoise_gaussian_filter(noisy_gray: np.ndarray) -> np.ndarray:
    """Gaussian smoothing on grayscale, sigma_g=1.0 per P6 paper."""
    return ndi.gaussian_filter(noisy_gray, sigma=1.0, mode='reflect')


# === Deep denoisers (operate on RGB, sliding window 128/96 to match training pipeline) ===

def denoise_deep_rgb(noisy_rgb: np.ndarray, method: str, sigma: int,
                     patch_size: int = 128, stride: int = 96) -> np.ndarray:
    """
    Apply deep denoiser via sliding window + overlap averaging.
    Mirrors training-time denoise_with_model() pattern (training nb Cell 10).
    Input/output: RGB float64 [0, 255], shape (H, W, 3).
    """
    model = denoise_models[method][sigma]  # nn.Module, already eval+device (Cell 4)
    h, w, c = noisy_rgb.shape

    # Pad to next multiple of patch_size, reflect mode (same as training)
    pad_h = (patch_size - h % patch_size) % patch_size
    pad_w = (patch_size - w % patch_size) % patch_size
    img_padded = np.pad(noisy_rgb, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')

    ph, pw = img_padded.shape[:2]
    result = np.zeros((ph, pw, c), dtype=np.float64)
    count  = np.zeros((ph, pw, 1), dtype=np.float64)

    with torch.no_grad():
        for y in range(0, ph - patch_size + 1, stride):
            for x in range(0, pw - patch_size + 1, stride):
                patch = img_padded[y:y+patch_size, x:x+patch_size] / 255.0  # [0,1]
                patch_t = (torch.from_numpy(patch).float()
                            .permute(2, 0, 1).unsqueeze(0).to(device))  # (1, 3, 128, 128)
                out = model(patch_t)
                if isinstance(out, (tuple, list)):
                    out = out[0]
                out_np = (out.squeeze(0).cpu().numpy()
                            .transpose(1, 2, 0)) * 255.0  # (128, 128, 3) in [0, 255]
                result[y:y+patch_size, x:x+patch_size] += out_np
                count[y:y+patch_size, x:x+patch_size]  += 1

    count = np.maximum(count, 1)
    result = result / count
    return np.clip(result[:h, :w], 0, 255)


# === Dispatcher: always returns GRAYSCALE denoised (for downstream BFPI) ===

def apply_denoiser(method: str, noisy_gray: np.ndarray, noisy_rgb: np.ndarray,
                   sigma: int) -> np.ndarray:
    """
    Return grayscale denoised image regardless of which method.
    - bm3d / gaussian_filter: operate on noisy_gray directly
    - deep methods: operate on noisy_rgb, project output via luminance
    """
    if method == 'bm3d':
        return denoise_bm3d(noisy_gray, sigma)
    elif method == 'gaussian_filter':
        return denoise_gaussian_filter(noisy_gray)
    elif method in ('autoencoder', 'dncnn', 'cae_pso'):
        denoised_rgb = denoise_deep_rgb(noisy_rgb, method, sigma)
        return rgb_to_luminance(denoised_rgb)
    raise ValueError(f'Unknown method: {method}')


# === HARD-ASSERT smoke test: all 5 methods MUST work before full sweep ===
# Per Standards E1 — physical invariant (model callable + shape match) is hard-assert.
# Aborts notebook here rather than silently logging warnings during 4-hour sweep.
print('Smoke test on 1 PnPLO image (HARD ASSERT — sweep aborts if any method fails):')
sample_name = next(iter(clean_images['PnPLO']))
sample_gray = clean_images['PnPLO'][sample_name]
sample_rgb  = clean_images_rgb['PnPLO'][sample_name]
print(f'  Sample: {sample_name}, gray={sample_gray.shape}, rgb={sample_rgb.shape}')

test_sigma = 20
noisy_gray = add_gaussian_noise(sample_gray, test_sigma, sample_name)
noisy_rgb  = add_gaussian_noise(sample_rgb,  test_sigma, sample_name)
print(f'  ✓ Noise added (σ={test_sigma}), gray|noisy-clean|='
      f'{np.abs(noisy_gray - sample_gray).mean():.2f}, '
      f'rgb|noisy-clean|={np.abs(noisy_rgb - sample_rgb).mean():.2f}')

for method in ['bm3d', 'gaussian_filter', 'autoencoder', 'dncnn', 'cae_pso']:
    t0 = time.time()
    out = apply_denoiser(method, noisy_gray, noisy_rgb, test_sigma)
    assert out.shape == sample_gray.shape, \
        f'{method} output shape {out.shape} != expected gray {sample_gray.shape}'
    assert 0 <= out.min() and out.max() <= 255, \
        f'{method} output out of [0,255]: [{out.min():.2f}, {out.max():.2f}]'
    print(f'  ✓ {method}: {time.time()-t0:.2f}s, range=[{out.min():.1f}, {out.max():.1f}]')

print('\n✓ All 5 methods pass smoke test — safe to proceed to Cell 8')


---
## Cell 7 — Idempotent guard (Rule D3)

If output CSV already exists and is healthy, skip the heavy compute loop. To force re-run: delete the CSV.

In [ ]:
# Cell 7: Idempotent guard (per-dataset) — P8 v5 (SCHEMA-AWARE)
# v5: skip a dataset only if its CSV is complete AND already carries the v0.3
#     phase-aware column 'rho_phase_1'. A v4-schema CSV (no rho_phase) is
#     treated as stale and RECOMPUTED -- this prevents the v3->v4 skip trap.
DATASETS_TO_SKIP = set()
expected_per_dataset = len(DENOISE_METHODS) * len(SIGMAS) * N_IMAGES_PER_DATASET
REQUIRED_V5_COL = 'rho_phase_1'

for ds_name, csv_path in OUTPUT_CSVS.items():
    if not csv_path.exists():
        print(f'  [{ds_name}] no existing output \u2192 will compute')
        continue
    try:
        df_existing = pd.read_csv(csv_path)
        n_rows = len(df_existing)
        size_kb = csv_path.stat().st_size / 1024
        has_v5 = REQUIRED_V5_COL in df_existing.columns
        if n_rows >= expected_per_dataset * 0.95 and has_v5:
            print(f'  \u2705 [{ds_name}] complete (v5): {n_rows}/{expected_per_dataset} '
                  f'rows, {size_kb:.1f} KB')
            print(f'     To force re-compute: !rm "{csv_path}"')
            DATASETS_TO_SKIP.add(ds_name)
        elif not has_v5:
            print(f'  \u267b\ufe0f  [{ds_name}] exists but is v4 schema '
                  f'(no {REQUIRED_V5_COL}) \u2192 will recompute for v5')
        else:
            print(f'  \u26a0\ufe0f  [{ds_name}] partial: {n_rows}/{expected_per_dataset} '
                  f'rows \u2192 will recompute')
    except Exception as e:
        print(f'  \u26a0\ufe0f  [{ds_name}] unreadable ({e}) \u2192 will overwrite')

print(f'\nWill skip {len(DATASETS_TO_SKIP)}/{len(DATASETS)} datasets')


---
## Cell 8 — Compute ρ_k batch loop (Phase A)

Heavy compute cell. Iterate over (dataset, sigma, method, image), compute ρ_k, accumulate to DataFrame. Save after each (dataset, sigma) block for resumability.

Expected time: ~30 min on A100 (BM3D is the bottleneck at ~3s/image CPU).

In [ ]:
# Cell 8: Compute rho_k (P8 v5 — adds rho_phase_k, the phase-aware Theorem 1 quantity)
from tqdm import tqdm

boundaries = get_band_boundaries(K=K_BANDS, scheme='linear')
print(f'K={K_BANDS} bands at boundaries: '
      f'{[("inf" if np.isinf(b) else f"{b/np.pi:.2f}pi") for b in boundaries]}')

radial_cache = {}
t_start = time.time()

for ds_name, ds_imgs_gray in clean_images.items():
    if ds_name in DATASETS_TO_SKIP:
        print(f'\n[{ds_name}] SKIPPED (already complete per Cell 7)')
        continue

    ds_imgs_rgb = clean_images_rgb[ds_name]
    output_csv = OUTPUT_CSVS[ds_name]
    rows = []

    for sigma in SIGMAS:
        print(f'\n[{ds_name}, \u03c3={sigma}] processing {len(ds_imgs_gray)} '
              f'images x {len(DENOISE_METHODS)} methods...')
        t_block = time.time()

        for img_name, clean_gray in tqdm(ds_imgs_gray.items(), desc=f'{ds_name} \u03c3={sigma}'):
            clean_rgb = ds_imgs_rgb[img_name]
            shape = clean_gray.shape[:2]
            if shape not in radial_cache:
                radial_cache[shape] = radial_frequency_map(*shape)
            radial_map = radial_cache[shape]

            # Clean-image band energies — invariant of method.
            E_clean = band_energy(clean_gray, boundaries, radial_map)

            noisy_gray = add_gaussian_noise(clean_gray, sigma, img_name)
            noisy_rgb  = add_gaussian_noise(clean_rgb,  sigma, img_name)

            for method in DENOISE_METHODS:
                try:
                    denoised = apply_denoiser(method, noisy_gray, noisy_rgb, sigma)
                    rho_sim   = compute_rho_k(clean_gray, denoised, boundaries,
                                              radial_map=radial_map, mode='sim')
                    rho_ratio = compute_rho_k(clean_gray, denoised, boundaries,
                                              radial_map=radial_map, mode='ratio')
                    # P8 v0.3 — phase-aware preservation (Theorem 1 quantity)
                    rho_phase = compute_rho_phase(clean_gray, denoised, boundaries,
                                                  radial_map=radial_map, E_clean=E_clean)
                    row = {
                        'dataset': ds_name, 'sigma': sigma, 'method': method,
                        'image': img_name,
                    }
                    for k in range(K_BANDS):
                        row[f'rho_{k+1}']         = float(rho_sim[k])
                        row[f'rho_ratio_{k+1}']   = float(rho_ratio[k])
                        row[f'rho_phase_{k+1}']   = float(rho_phase[k])
                        row[f'E_band{k+1}_clean'] = float(E_clean[k])
                    rows.append(row)
                except Exception as e:
                    print(f'  \u26a0\ufe0f  {method} failed on {img_name}: '
                          f'{type(e).__name__}: {e}')

        elapsed = time.time() - t_block
        print(f'  Block done in {elapsed:.1f}s. '
              f'Total elapsed: {(time.time()-t_start)/60:.1f} min')

        df_partial = pd.DataFrame(rows)
        df_partial.to_csv(output_csv, index=False)
        print(f'  \U0001f4be Saved partial: {len(df_partial)} rows to {output_csv.name}')

    print(f'\n\u2705 [{ds_name}] complete: {len(rows)} rows \u2192 {output_csv}')

print(f'\n\u2705 All datasets done in {(time.time()-t_start)/60:.1f} min')
print('   Schema v5: rho_k (energy sim, descriptive) + rho_ratio_k (diagnostic) '
      '+ rho_phase_k (PHASE-AWARE, Theorem 1) + E_band{k}_clean')


---
## Cell 9 — Verify saved output schema (Rule A5)

Future Notebook 3 will consume this CSV. We must verify schema is well-formed.

In [ ]:
# Cell 9: Verify saved output (P8 v1.1 — load 3 per-dataset CSVs and merge for verification)
df_list = []
for ds_name, csv_path in OUTPUT_CSVS.items():
    if not csv_path.exists():
        print(f'  ⚠️  [{ds_name}] CSV not found: {csv_path.name} — skipped')
        continue
    df_ds = pd.read_csv(csv_path)
    print(f'  ✓ [{ds_name}] loaded {len(df_ds)} rows from {csv_path.name}')
    df_list.append(df_ds)

assert df_list, 'FAIL: no CSV files loaded — Cell 8 may not have run successfully'
df = pd.concat(df_list, ignore_index=True)
print(f'\nMerged: {df.shape}')
print(f'  Columns: {list(df.columns)}')
print(f'  Datasets: {sorted(df["dataset"].unique())}')
print(f'  Methods: {sorted(df["method"].unique())}')
print(f'  Sigmas: {sorted(df["sigma"].unique())}')

# Verify required columns
required_cols = ['dataset', 'sigma', 'method', 'image'] + [f'rho_{k+1}' for k in range(K_BANDS)]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f'FAIL: missing columns {missing}'

# Verify physical invariants (Rule E1)
for k in range(K_BANDS):
    col = f'rho_{k+1}'
    in_range = ((df[col] >= 0) & (df[col] <= 1)).mean()
    assert in_range > 0.99, f'FAIL: {col} has {(1-in_range)*100:.1f}% values outside [0,1]'
print(f'\n✓ All rho values in [0,1] (sim mode invariant)')

# Print per-cell counts
print('\nPer-cell row counts:')
counts = df.groupby(['dataset', 'sigma', 'method']).size().unstack(fill_value=0)
print(counts)

---
## Cell 10 — Sanity check against P6 paper

P6 reported BM3D at σ=20 on PnPLO: SPS-HF = 0.87 (high-freq retention).
In P8 sim mode with K=4, we expect ρ_3 (mid-high band [π/2, 3π/4]) for BM3D ≈ 0.85±0.10.

In [ ]:
# Cell 10: Sanity check (P8 v4 — internal P8 consistency, no external P6 anchor)
# Rationale: P8 ρ_3 ∈ [0.5π, 0.75π] ≠ P6 ρ_high ∈ [0.5π, π]; comparison was apples vs oranges.
import pandas as pd

print('=== INTERNAL SANITY CHECKS (P8 self-validation) ===\n')

# Reload 3 CSVs
csvs = [pd.read_csv(OUTPUT_CSVS[ds]) for ds in DATASETS.keys()]
df = pd.concat(csvs, ignore_index=True)
print(f'Loaded {len(df)} rows across {df["dataset"].nunique()} datasets')
print(f'Methods present: {sorted(df["method"].unique())}\n')

# === Check 1: BM3D cross-dataset stability (replaces old P6 anchor) ===
print('[Check 1] BM3D ρ_3 cross-dataset CV at σ=20 (expect < 5% — universal characterizability):')
piv = df[(df['method']=='bm3d') & (df['sigma']==20)].groupby('dataset')['rho_3'].mean()
cv = piv.std() / piv.mean() * 100
print(f'  ρ_3 by dataset: {piv.round(3).to_dict()}')
print(f'  CV = {cv:.2f}%  →  {"✓ PASS — BM3D is dataset-invariant anchor" if cv < 5 else "⚠️  DEVIATES"}')

# === Check 2: ρ_sim ∈ [0, 1] mathematical invariant ===
print('\n[Check 2] ρ_sim ∈ [0, 1] (theorem invariant):')
all_pass = True
for k in [1,2,3,4]:
    col = f'rho_{k}'
    n_out = ((df[col] < -1e-6) | (df[col] > 1 + 1e-6)).sum()
    status = '✓' if n_out == 0 else '❌'
    print(f'  {col}: {n_out} out-of-range rows  {status}')
    if n_out > 0:
        all_pass = False

# === Check 3: BM3D ρ_k(σ) monotone decreasing — physically expected ===
print('\n[Check 3] BM3D ρ_3 monotone ↓ as σ↑ (physical: more noise → less preservation):')
piv = df[df['method']=='bm3d'].groupby('sigma')['rho_3'].mean()
diffs = piv.diff().dropna()
mono = (diffs < 0).all()
print(f'  ρ_3 by σ: {piv.round(3).to_dict()}')
print(f'  Monotone decreasing: {"✓ PASS" if mono else "⚠️  NOT MONOTONE"}')

# === Check 4: Two-mode diagnostic — rho_ratio columns + artifact mode count ===
print('\n[Check 4] Two-mode diagnostic (rho_ratio columns + artifact mode > 1):')
ratio_cols_present = all(f'rho_ratio_{k}' in df.columns for k in [1,2,3,4])
if not ratio_cols_present:
    print('  ❌ MISSING rho_ratio columns — v4 patch not applied?')
else:
    print('  ✓ rho_ratio columns present')
    for k in [1,2,3,4]:
        col = f'rho_ratio_{k}'
        n_artifact = (df[col] > 1.0).sum()
        n_oversmooth = (df[col] < 0.1).sum()
        print(f'  {col}: min={df[col].min():.3f}, max={df[col].max():.3f}, '
              f'artifact-mode (>1)={n_artifact}, over-smooth-mode (<0.1)={n_oversmooth}')

# === Check 5: Band energies sum approximately to image variance (Parseval) ===
print('\n[Check 5] Band energies E_k present (for P6-equivalent aggregation):')
energy_cols_present = all(f'E_band{k}_clean' in df.columns for k in [1,2,3,4])
if energy_cols_present:
    # Sample 1 image: sum of band energies ≈ total spectral energy
    sample_row = df[df['method']=='bm3d'].iloc[0]
    E_total = sum(sample_row[f'E_band{k}_clean'] for k in [1,2,3,4])
    print(f'  ✓ Energy columns present. Sample image total E = {E_total:.2e}')
    # Compute P6-equivalent rho_high (weighted avg of band 3 + band 4)
    df['rho_high_P6eq'] = (
        (df['E_band3_clean'] * df['rho_ratio_3'] + df['E_band4_clean'] * df['rho_ratio_4']) /
        (df['E_band3_clean'] + df['E_band4_clean'])
    )
    bm3d_high = df[(df['method']=='bm3d') & (df['sigma']==20)
                    & (df['dataset']=='PnPLO')]['rho_high_P6eq'].mean()
    print(f'  → BM3D ρ_high_P6eq (PnPLO, σ=20) = {bm3d_high:.3f}  (legacy P6 anchor was ~0.85)')
else:
    print('  ❌ MISSING E_band columns — v4 patch not applied?')

print('\n=== End sanity checks ===')
print('NOTE: P8 ρ_k is a refined quantity, not equivalent to P6 ρ_high.')
print('      The ρ_high_P6eq column allows backward-compatible comparison if needed.')


---
## Cell 11 — Phase A summary + decision point

User reviews this output to decide whether to:
- (a) Proceed to Notebook 2 (estimate α_k)
- (b) Run Phase B (full test sets, ~5h)
- (c) Debug Phase A first (anchor failed, ordering wrong)

In [ ]:
# Cell 11: Summary
print('=' * 70)
print('PHASE A SUMMARY')
print('=' * 70)
print('Output files (per-dataset):')
for ds_name, csv_path in OUTPUT_CSVS.items():
    if csv_path.exists():
        n = len(pd.read_csv(csv_path))
        print(f'  ✓ [{ds_name}] {n:>6d} rows → {csv_path}')
    else:
        print(f'  ❌ [{ds_name}] MISSING: {csv_path}')

print(f'\nMerged total: {len(df)} rows')
print(f'Coverage: {df["dataset"].nunique()} datasets × {df["sigma"].nunique()} sigmas × {df["method"].nunique()} methods × ~{N_IMAGES_PER_DATASET} images')

print(f'\n=== DECISION POINT ===')
print('Next steps (Barry to decide based on these results):')
print('  (a) PROCEED to Notebook 2 (estimate α_k for YOLOv9m + v10m on PnPLO)')
print('  (b) RUN PHASE B (full train sets — set N_IMAGES_PER_DATASET = None, delete the 3 CSVs)')
print('  (c) DEBUG if anchor failed or ordering wrong')

---
## Cell 12 — (Optional) Phase B full set

To run Phase B:
1. Set `N_IMAGES_PER_DATASET = None` in Cell 3
2. Delete the 3 per-dataset CSVs to bypass idempotent guard:
   ```
   !rm "{PnPLO output_csv path}"
   !rm "{VOC output_csv path}"
   !rm "{INRIA output_csv path}"
   ```
3. Re-run from Cell 5

Expected compute: ~5-8 hours on A100 (PnPLO 944 + VOC 6340 + INRIA 632 = 7916 imgs × 5 methods × 5 sigma). BM3D is the bottleneck. Consider chunking via Drive resumability — Cell 8 already saves after each (dataset, sigma) block.

In [ ]:
# Cell 12: Phase B placeholder
print('Phase B not run by default — see markdown instructions above.')
print('To run: edit Cell 3 (N_IMAGES_PER_DATASET=None), delete the 3 per-dataset CSVs, re-run from Cell 5.')
print()
print('Current Phase A CSVs:')
for ds_name, csv_path in OUTPUT_CSVS.items():
    if csv_path.exists():
        print(f'  rm "{csv_path}"')

# Paper figures — IEEE TIP standard (P8 v5)

Cells 13-17 export the publication figures. Run them **after** the v5 compute (Cells 7-11) has produced the `rho_phase_*` columns.

Output: vector PDFs in `P8_outputs/paper_figures/` on Drive — download them into the Overleaf `figures/` folder.

| file | paper slot |
|---|---|
| `P8_fig_dssp_alpha.pdf` | Section 5 — DSSP profile |
| `P8_fig_twomode.pdf` | Section 3.5 / 5 — two-mode diagnostic |
| `P8_fig_crossdataset.pdf` | Section 5 — cross-dataset stability |
| `P8_fig_phase_vs_energy.pdf` | Section 3 — phase-aware vs energy |

Theorem-1 verification and the bound-ranking figure are produced by Notebook 3.

In [ ]:
# Cell 13: Paper-figure setup — IEEE TIP standard (P8 v5)
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import yaml

# --- IEEE TIP figure style: vector PDF, embedded serif fonts ---
mpl.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8,
    'legend.fontsize': 6.5, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'axes.linewidth': 0.6, 'lines.linewidth': 1.0,
    'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'pdf.fonttype': 42, 'ps.fonttype': 42,            # embed TrueType (IEEE requires)
    'savefig.dpi': 600, 'savefig.bbox': 'tight',
})
COL1, COL2 = 3.5, 7.16          # IEEE TIP single- / double-column width (inches)

# Okabe-Ito colorblind-safe palette (also separable in grayscale print)
OI = {'orange': '#E69F00', 'sky': '#56B4E9', 'green': '#009E73',
      'blue': '#0072B2', 'verm': '#D55E00', 'purple': '#CC79A7'}
METHOD_COLOR = {'bm3d': OI['blue'], 'dncnn': OI['verm'], 'autoencoder': OI['green'],
                'cae_pso': OI['orange'], 'gaussian_filter': OI['purple']}
METHOD_LABEL = {'bm3d': 'BM3D', 'dncnn': 'DnCNN', 'autoencoder': 'AE',
                'cae_pso': 'CAE+PSO', 'gaussian_filter': 'Gaussian'}
METHOD_ORDER = ['bm3d', 'dncnn', 'autoencoder', 'cae_pso', 'gaussian_filter']
BANDS = [1, 2, 3, 4]
BAND_LABEL = ['Band 1\n[0,$\\pi$/4)', 'Band 2\n[$\\pi$/4,$\\pi$/2)',
              'Band 3\n[$\\pi$/2,3$\\pi$/4)', 'Band 4\n[3$\\pi$/4,$\\infty$)']

# --- output dir: download these PDFs into the Overleaf  figures/  folder ---
FIG_DIR = DRIVE_ROOT / 'P8_outputs' / 'paper_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# --- DSSP alpha profile from Notebook 2 ---
# Path MUST match Notebook 2 ALPHA_YAML_OUT (and Notebook 3 ALPHA_YAML_PATH):
#   DRIVE_ROOT/04_RESULT_TRAIN_KARTHY/YOLO_Denoise_Experiment_Karthy/P8_outputs/
ALPHA_YAML = (DRIVE_ROOT / '04_RESULT_TRAIN_KARTHY' / 'YOLO_Denoise_Experiment_Karthy'
              / 'P8_outputs' / 'alpha_profile_yolov8m_pnplo.yaml')
if not ALPHA_YAML.exists():
    raise FileNotFoundError(
        f'alpha profile not found: {ALPHA_YAML}\n'
        f'Run Notebook 2 first. This path must match NB2 ALPHA_YAML_OUT -- '
        f'there is no hardcoded fallback, which would silently use stale alpha.')
_a = yaml.safe_load(open(ALPHA_YAML))['alpha']
ALPHA = [_a[f'band_{k}'] for k in BANDS]
print(f'\u2713 alpha loaded from {ALPHA_YAML.name}: {[round(x, 3) for x in ALPHA]}')

# --- merged spectral data ---
df_fig = pd.concat([pd.read_csv(OUTPUT_CSVS[d]) for d in OUTPUT_CSVS], ignore_index=True)
HAS_PHASE = 'rho_phase_1' in df_fig.columns
print(f'\u2713 figure data: {df_fig.shape}  | phase-aware columns present: {HAS_PHASE}')
if not HAS_PHASE:
    print('  \u2192 Fig C uses energy-rho fallback; Fig D is skipped. '
          'Re-run the v5 compute for the paper figures.')

def save_fig(fig, name):
    p = FIG_DIR / f'{name}.pdf'
    fig.savefig(p)
    print(f'  saved \u2192 {p}')
    plt.show()


In [ ]:
# Cell 14: Figure — DSSP alpha profile (YOLOv8m)   [paper Section 5]
fig, ax = plt.subplots(figsize=(COL1, 2.3))
ax.bar(range(4), ALPHA, width=0.62,
       color=[OI['blue'], OI['sky'], OI['orange'], OI['verm']],
       edgecolor='black', linewidth=0.5)
for i, v in enumerate(ALPHA):
    ax.text(i, v + 0.012, f'{v:.3f}', ha='center', fontsize=7)
ax.set_xticks(range(4))
ax.set_xticklabels(BAND_LABEL, fontsize=6.3)
ax.set_ylabel(r'DSSP weight  $\alpha_k$')
ax.set_ylim(0, max(ALPHA) * 1.25)
ax.set_axisbelow(True)
ax.grid(axis='y', lw=0.4, alpha=0.5)
ax.annotate(f'low-frequency dominance\n$\\alpha_1+\\alpha_2$ = {ALPHA[0]+ALPHA[1]:.2f}',
            xy=(0.5, ALPHA[0] * 0.82), xytext=(1.85, max(ALPHA) * 0.86),
            fontsize=6.5, ha='left', arrowprops=dict(arrowstyle='-', lw=0.5))
save_fig(fig, 'P8_fig_dssp_alpha')


In [ ]:
# Cell 15: Figure — two-mode failure diagnostic (energy ratio, log scale)
#          [paper Section 3.5 / Section 5]
fig, ax = plt.subplots(figsize=(COL2, 2.7))
nm = len(METHOD_ORDER)
bw = 0.74 / nm
ax.axhspan(1e-3, 0.5, color=OI['sky'], alpha=0.10)     # over-smoothing zone
ax.axhspan(1.5, 2e2, color=OI['verm'], alpha=0.10)     # artifact-injection zone
for mi, m in enumerate(METHOD_ORDER):
    data = [df_fig[df_fig.method == m][f'rho_ratio_{k}'].values for k in BANDS]
    pos = [bi + (mi - (nm - 1) / 2) * bw for bi in range(4)]
    bp = ax.boxplot(data, positions=pos, widths=bw * 0.86, patch_artist=True,
                    showfliers=False, medianprops=dict(color='black', lw=0.8),
                    whiskerprops=dict(lw=0.5), capprops=dict(lw=0.5),
                    boxprops=dict(lw=0.4))
    for b in bp['boxes']:
        b.set_facecolor(METHOD_COLOR[m])
ax.set_yscale('log')
ax.set_ylim(1e-3, 2e2)
ax.set_xlim(-0.6, 3.6)
ax.axhline(1.0, color='black', lw=0.6, ls='--')
ax.axhline(0.5, color=OI['sky'], lw=0.7)
ax.axhline(1.5, color=OI['verm'], lw=0.7)
ax.text(-0.5, 0.011, 'over-smoothing', fontsize=6, color=OI['blue'], ha='left')
ax.text(-0.5, 55, 'artifact injection', fontsize=6, color=OI['verm'], ha='left')
ax.set_xticks(range(4))
ax.set_xticklabels(BAND_LABEL, fontsize=6.3)
ax.set_ylabel(r'energy ratio  $\rho^{\mathrm{ratio}}_k$')
ax.legend(handles=[Patch(facecolor=METHOD_COLOR[m], edgecolor='black', lw=0.4,
                         label=METHOD_LABEL[m]) for m in METHOD_ORDER],
          ncol=5, loc='upper center', bbox_to_anchor=(0.5, 1.13),
          frameon=False, handlelength=1.2)
save_fig(fig, 'P8_fig_twomode')


In [ ]:
# Cell 16: Figure — cross-dataset stability of band-3 preservation   [paper Section 5]
RHO3 = 'rho_phase_3' if HAS_PHASE else 'rho_3'
if not HAS_PHASE:
    print('  \u26a0\ufe0f  using energy-rho fallback (rho_3); re-run v5 for the '
          'phase-aware paper figure.')
DS = list(OUTPUT_CSVS.keys())
_pal = [OI['blue'], OI['orange'], OI['green']]
_mk = ['o', 's', '^']
DCOL = {d: _pal[i] for i, d in enumerate(DS)}
DMK = {d: _mk[i] for i, d in enumerate(DS)}
fig, axs = plt.subplots(1, 2, figsize=(COL2, 2.5), sharey=True)
for ax, m in zip(axs, ['bm3d', 'dncnn']):
    piv = df_fig[df_fig.method == m].pivot_table(
        index='sigma', columns='dataset', values=RHO3, aggfunc='mean')
    for d in DS:
        ax.plot(piv.index, piv[d], marker=DMK[d], ms=3.5, color=DCOL[d],
                label=d, lw=1.1)
    cv = (piv.std(axis=1) / piv.mean(axis=1) * 100)
    ax.set_title(f'{METHOD_LABEL[m]}   (cross-dataset CV '
                 f'{cv.min():.0f}\u2013{cv.max():.0f}%)', fontsize=7.5)
    ax.set_xlabel(r'noise level  $\sigma$')
    ax.set_xticks(sorted(df_fig.sigma.unique()))
    ax.grid(lw=0.4, alpha=0.5)
    ax.set_axisbelow(True)
axs[0].set_ylabel(r'band-3 preservation  $\rho_3$')
axs[0].set_ylim(top=1.05)
axs[0].legend(frameon=False, handlelength=1.4, loc='lower left')
save_fig(fig, 'P8_fig_crossdataset')


In [ ]:
# Cell 17: Figure — phase-aware vs energy-only preservation   [paper Section 3]
if not HAS_PHASE:
    print('  \u26a0\ufe0f  Fig D needs the v5 phase-aware data (rho_phase_k). '
          'Re-run the v5 compute, then run this cell.')
else:
    fig, ax = plt.subplots(figsize=(COL1, 3.0))
    bcol = [OI['blue'], OI['sky'], OI['orange'], OI['verm']]
    for k in BANDS:
        ax.scatter(df_fig[f'rho_{k}'], df_fig[f'rho_phase_{k}'], s=3, alpha=0.35,
                   color=bcol[k - 1], edgecolors='none', label=f'Band {k}',
                   rasterized=True)
    lim_lo = min(-1.0, float(df_fig[[f'rho_phase_{k}' for k in BANDS]].min().min()))
    ax.plot([lim_lo, 1], [lim_lo, 1], color='black', lw=0.7, ls='--', label='$y=x$')
    ax.axhline(0, color='grey', lw=0.5)
    ax.set_xlabel(r'energy-only preservation  $\rho_k^{\mathrm{energy}}$ (v4)')
    ax.set_ylabel(r'phase-aware preservation  $\rho_k$ (Theorem 1)')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(lim_lo, 1.05)
    ax.set_axisbelow(True)
    ax.grid(lw=0.4, alpha=0.4)
    ax.legend(frameon=False, markerscale=2.5, handlelength=1.2, loc='lower right')
    save_fig(fig, 'P8_fig_phase_vs_energy')
